Statistical Engine
Build the core analysis logic first before touching any LLM:

1. Ingest a CSV with control/variant metrics
2. Calculate p-values, confidence intervals, relative lift
3. Flag underpowered tests (sample size checks)
4. Output a structured result dictionary

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats

### 1. Ingest a CSV with control/variant metrics

In [47]:
df = pd.read_csv('../data/ab_test_experiments_v2.csv')
df.head(3)

,test_id,test_name,product_area,hypothesis,primary_metric,secondary_metric,device_type,audience_segment,start_date,end_date,...,relative_lift_pct,secondary_metric_control,secondary_metric_variant,avg_order_value,control_revenue,variant_revenue,revenue_delta,outcome_type,analyst,notes
0,EXP-1000,Category Filter UX v1,Acquisition,Clearer value proposition will improve activation,signup_rate,bounce_rate,Mobile,Returning Users,2023-10-15,2023-10-23,...,7.46,0.4066,0.3666,407.76,92969.28,54639.84,-38329.44,underpowered,Sam T.,Test targeting returning users on mobile. Hypo...
1,EXP-1001,Trust Badge Placement v2,Onboarding,Reducing friction will increase conversion,signup_rate,avg_session_duration,Desktop,Free Tier,2023-05-31,2023-07-12,...,14.69,234.1898,268.4530,431.28,5879640.24,2889144.72,-2990495.52,clear_winner,Sam T.,Test targeting free tier on desktop. Hypothesi...
2,EXP-1002,Progress Bar Addition v3,Onboarding,Urgency messaging will drive faster decisions,signup_rate,pages_per_session,Mobile,Premium Members,2023-04-23,2023-06-04,...,12.14,0.4045,0.4046,56.44,3447129.44,1656683.32,-1790446.12,clear_winner,Alex R.,Test targeting premium members on mobile. Hypo...


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   test_id                    200 non-null    object 
 1   test_name                  200 non-null    object 
 2   product_area               200 non-null    object 
 3   hypothesis                 200 non-null    object 
 4   primary_metric             200 non-null    object 
 5   secondary_metric           200 non-null    object 
 6   device_type                200 non-null    object 
 7   audience_segment           200 non-null    object 
 8   start_date                 200 non-null    object 
 9   end_date                   200 non-null    object 
 10  test_duration_days         200 non-null    int64  
 11  traffic_split_control_pct  200 non-null    int64  
 12  test_direction             200 non-null    object 
 13  test_tails                 200 non-null    object 

In [10]:
df.outcome_type.unique()

array(['clear_winner', 'clear_loser', 'neutral', 'novelty_effect',
       'underpowered', 'false_positive'], dtype=object)

### 2. Calculate p-values, confidence intervals, relative lift

In [59]:
def calculate_pvalue(control_visitors, variant_visitors, 
                     control_conversions, variant_conversions, 
                     test_tails, test_direction):
    # Conversion rates
    p_c = control_conversions / control_visitors
    p_v = variant_conversions / variant_visitors
 
    # Pooled proportion & standard error
    p_pool = (control_conversions + variant_conversions) / (control_visitors + variant_visitors)
    se = (p_pool * (1 - p_pool) * (1 / control_visitors + 1 / variant_visitors))**0.5
    
    # z-statistic (variant - control direction)
    z_stat = (p_v - p_c) / se
    # two tailed p-value calculation
    if test_tails == 'two_tailed':
        p_value = 2*(1-stats.norm.cdf(abs(z_stat)))
    else:
        if test_direction in ("superiority", "non_inferiority"):
            p_value = 1 - stats.norm.cdf(z_stat)   # P(Z > z_stat)
        else:
            p_value = stats.norm.cdf(z_stat)        # P(Z < z_stat)    
    
    return round(p_value, 4)

In [60]:
calculate_pvalue(1406, 769, 228, 134,  'one_tailed', 'superiority') #underpowered

0.2346

In [61]:
calculate_pvalue(215388,	92310,	13633,	6699,	'one_tailed',	'superiority') #winnter

0.0

In [103]:
def add_pvalues(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for i,row in df.iterrows():
        res = calculate_pvalue(
                control_conversions=int(row["control_conversions"]),
                control_visitors=int(row["control_visitors"]),
                variant_conversions=int(row["variant_conversions"]),
                variant_visitors=int(row["variant_visitors"]),
                test_tails=row["test_tails"],
                test_direction=row["test_direction"],
            )
        results.append(res)
    result_df = pd.DataFrame(results, columns = ['p_values'])
    out = df.copy()
    out['p_values'] = result_df['p_values']
    return out
    

In [104]:
test_df = add_pvalues(df)

In [107]:
test_df.columns

Index(['test_id', 'test_name', 'product_area', 'hypothesis', 'primary_metric',
       'secondary_metric', 'device_type', 'audience_segment', 'start_date',
       'end_date', 'test_duration_days', 'traffic_split_control_pct',
       'test_direction', 'test_tails', 'alpha', 'mde_relative_pct',
       'control_visitors', 'variant_visitors', 'total_visitors',
       'control_conversions', 'variant_conversions', 'control_conversion_rate',
       'variant_conversion_rate', 'relative_lift_pct',
       'secondary_metric_control', 'secondary_metric_variant',
       'avg_order_value', 'control_revenue', 'variant_revenue',
       'revenue_delta', 'outcome_type', 'analyst', 'notes', 'p_values'],
      dtype='object')

In [ ]:
def calculate_pvalue(
    control_conversions: int,
    control_visitors: int,
    variant_conversions: int,
    variant_visitors: int,
    test_tails: str = "two_tailed",
    test_direction: str = "superiority",
) -> dict:
    """
    Calculate the p-value for an A/B test using a two-proportion z-test
    with pooled variance.
 
    Parameters
    ----------
    control_conversions : int   – number of conversions in the control group
    control_visitors    : int   – total visitors in the control group
    variant_conversions : int   – number of conversions in the variant group
    variant_visitors    : int   – total visitors in the variant group
    test_tails          : str   – 'one_tailed' or 'two_tailed'
    test_direction      : str   – 'superiority'      → variant > control (right tail)
                                  'non_inferiority'   → variant >= control (right tail)
 
    Returns
    -------
    dict with keys:
        z_stat          : float  – z-statistic
        p_value         : float  – computed p-value
        test_tails      : str
        test_direction  : str
        control_rate    : float  – conversion rate for control
        variant_rate    : float  – conversion rate for variant
        relative_lift   : float  – (variant_rate - control_rate) / control_rate
    """
    # --- Input validation ---
    if control_visitors <= 0 or variant_visitors <= 0:
        raise ValueError("Visitor counts must be positive integers.")
    if not (0 <= control_conversions <= control_visitors):
        raise ValueError("control_conversions must be between 0 and control_visitors.")
    if not (0 <= variant_conversions <= variant_visitors):
        raise ValueError("variant_conversions must be between 0 and variant_visitors.")
 
    test_tails = test_tails.strip().lower()
    test_direction = test_direction.strip().lower()
 
    if test_tails not in ("one_tailed", "two_tailed"):
        raise ValueError("test_tails must be 'one_tailed' or 'two_tailed'.")
    if test_direction not in ("superiority", "non_inferiority"):
        raise ValueError("test_direction must be 'superiority' or 'non_inferiority'.")
 
    # --- Conversion rates ---
    p_c = control_conversions / control_visitors
    p_v = variant_conversions / variant_visitors
 
    # --- Pooled proportion & standard error ---
    p_pool = (control_conversions + variant_conversions) / (control_visitors + variant_visitors)
    se = math.sqrt(p_pool * (1 - p_pool) * (1 / control_visitors + 1 / variant_visitors))
 
    if se == 0:
        raise ValueError("Standard error is zero — check that conversion rates are not 0 or 1.")
 
    # --- Z-statistic (variant − control direction) ---
    z_stat = (p_v - p_c) / se
 
    # --- P-value ---
    if test_tails == "two_tailed":
        # H0: p_v == p_c   H1: p_v != p_c
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    else:
        # One-tailed: direction determines which tail
        # superiority / non_inferiority → variant should be HIGHER (right tail)
        if test_direction in ("superiority", "non_inferiority"):
            p_value = 1 - stats.norm.cdf(z_stat)   # P(Z > z_stat)
        else:
            p_value = stats.norm.cdf(z_stat)        # P(Z < z_stat)
 
    relative_lift = (p_v - p_c) / p_c if p_c > 0 else float("nan")
 
    return {
        "z_stat": round(z_stat, 4),
        "p_value": round(p_value, 6),
        "test_tails": test_tails,
        "test_direction": test_direction,
        "control_rate": round(p_c, 6),
        "variant_rate": round(p_v, 6),
        "relative_lift": round(relative_lift, 6),
    }

In [ ]:
# single tailed p-value calculation 


### 3. Flag underpowered tests (sample size checks)